<a href="https://colab.research.google.com/github/sivaagowsikan-arch/Statistical-Learning-e22115/blob/main/Assignment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comprehensive Data Sanitization, Processing, & Advanced Exploration Engine

## Overview
This production-grade Python toolkit is a modular, object-oriented framework designed to automate the initial 80% of data science workloads: data cleaning, exploration, and interactive visualization within Google Colab environments.

By encapsulating complex workflows inside structural `DataInspector` and `PlottingMethods` classes, the engine mitigates repetitive engineering tasks—such as diagnostic null tracking, outlier mitigation, non-linear feature scaling, and high-level multi-metric statistical association mapping—while remaining entirely resilient against runtime exceptions and structural data anomalies.

---

## Technical Features & Architectural Framework

### 1. Data Ingestion & Robust Sanitization
* **Native Colab Integration:** Utilizes an asynchronous `upload_data()` pipeline powered by `google.colab.files` and `io.BytesIO` streams for localized CSV asset ingestion.
* **Defensive Schema Remediation:** Automatically identifies and strips hidden duplicate column names to completely eliminate scalar shape mismatch runtime conflicts.
* **Garbage Token Normalization:** Scans and converts untyped noise strings (e.g., `'?'`, `'n/a'`, `'NULL'`, `' '`) into standard `np.nan` references.
* **Adaptive Type Coercion:** Dynamically forces text-represented attributes into numeric structures, validating evaluations via safe series masking to ensure columns are only cast if conversion does not yield an entirely null series.

### 2. Structural Diagnostics & Exploratory Cleaning
* **Comprehensive Metadata Summaries:** Provides row/column matrix dimensions, automated isolation of numeric versus categorical configurations, missing entry value trackers, and dynamic 20-row sample previews.
* **Intelligent Imputation Engine:** Supports multiple algorithmic missing-value remediation strategies, including localized column `mean`, `median`, `mode`, or fixed custom `constant` replacements applied safely in-place.
* **Statistical Outlier Trimming:** Implements robust Interquartile Range (IQR) boundary calculations ($[Q_1 - 1.5 \times IQR, Q_3 + 1.5 \times IQR]$) to track and systematically purge volatile extreme values.
* **Targeted Interactive Pruning:** Features user-prompted or parameter-driven string-parsed tools to safely delete arbitrary columns or target row indices without breaking execution order.

### 3. Machine Learning Preprocessing (Feature Engineering)
* **Numeric Scaling Architectures:** Features a suite of standard transformation utilities including **Min-Max**, **Standard Z-score**, and **Robust Median/IQR** scaling, all protected by a microscopic epsilon scalar ($+1e-9$) to prevent constant-column division-by-zero errors.
* **Categorical Encoding Suites:** Delivers seamless discrete conversions via multi-variate **One-Hot Encoding** (utilizing `drop_first=True` to prevent the dummy variable trap), **Ordinal Integer Map Encoding**, and **Uniform Rescaled Discrete Normalization**.
* **Unified Pipeline Merging:** Connects distinct numeric and categorical transformation arrays back into a synchronized, high-performance dataframe matrix via column-wise concatenation.

### 4. Interactive Plotly Visualization Layer
* **Univariate Subplots:** Consolidates single-variable continuous feature checks into a 3-panel subplot matrix tracking an interactive Horizontal Violin plot (with nested box visibility), a Sequence Index Scatter map (`Index vs Value`), and an optimal Frequency Histogram.
* **Discrete Frequency Analytics:** Generates multi-class categorical bar charts with automated percentage overlays that display absolute sample counts alongside their true relative proportions (`Count (Percentage%)`).
* **Smart Relationship Mapping:** Automatically runs an internal runtime type-check (`np.issubdtype`) to adaptively select and render the optimal chart configuration:
  * *Continuous vs. Continuous:* Scatter plot equipped with an Ordinary Least Squares (`OLS`) linear trendline.
  * *Categorical vs. Categorical:* Clustered grouped bar interaction chart.
  * *Mixed (Categorical vs. Continuous):* Multi-box comparative plots overlaying all active data distributions.

### 5. Advanced Statistical Association Mapping
* **Multi-Metric Relationship Heatmap:** Solves the traditional visual limitation of standard correlation matrices by building a unified heatmap that adaptively selects mathematical association metrics depending on target column interactions:
  * *Numeric vs. Numeric Pairs:* Standard **Pearson’s correlation coefficient ($r$)**.
  * *Categorical vs. Categorical Pairs:* **Cramér’s V association mapping** derived from raw $\chi^2$ contingency tables.
  * *Mixed (Categorical vs. Numeric) Pairs:* Non-linear **Eta-squared ($\eta^2$)** metrics powered by an internal One-Way Analysis of Variance (ANOVA) F-statistic calculation.
* **Array Dimension Isolation:** Implements defensive NumPy array flattening (`.flatten()`) across all lower-level correlation engines, completely preventing multidimensional Pandas DataFrame shape errors.

### 6. Isolated Custom Plotting Layer (`PlottingMethods`)
* **Decoupled Graphics Utility:** Houses standalone plotting routines fully isolated from the active workspace state to manage granular visual components including independent Bar, Pie, Histogram, Heatmap, Sankey, and Sunburst graphs.
* **HTML Component Delivery:** Wraps and outputs complex figures as clean, lightweight HTML script strings using Plotly's CDN integration—optimized for flexible dashboard embedding or direct UI presentation via notebook `display(HTML(...))`.

In [ ]:
import io
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.stats import pearsonr, chi2_contingency, f_oneway
from google.colab import files
from IPython.display import display, HTML

# Enforce explicit widget/renderer handling inside Google Colab environments
pio.renderers.default = "colab"


class DataInspector:
    """
    Modular Data Sanitization, Processing, & Advanced Exploration Engine.
    Fully optimized to run interactive workflows directly inside Google Colab environments.
    """

    def __init__(self, df: pd.DataFrame = None):
        self._df = None
        self.df = df  # Triggers the reactive property setter directly

    @property
    def df(self) -> pd.DataFrame:
        """Exposes the internal workspace data frame pattern safely."""
        return self._df

    @df.setter
    def df(self, value: pd.DataFrame):
        """Automatically sanitizes, coerces types, and deduplicates headers upon asset assignment."""
        self._df = value
        if self._df is not None:
            self._sanitize_and_coerce()

    def upload_data(self):
        """Triggers local file ingestion prompts safely within Google Colab environments."""
        print("Please upload your target CSV data asset:")
        uploaded = files.upload()
        if not uploaded:
            print("File selection sequence cancelled.")
            return

        file_name = list(uploaded.keys())[0]
        self.df = pd.read_csv(io.BytesIO(uploaded[file_name]))
        print(f"Data Successfully Loaded! Matrix Dimensions: {self.df.shape}")

    def _sanitize_and_coerce(self):
        """Internal helper to deduplicate columns, convert garbage strings to NaN, and optimize schemas."""
        if self._df is None:
            return

        # 1. Defensively handle and rename duplicate column headers to completely eliminate scalar conversion errors
        if self._df.columns.duplicated().any():
            new_cols = []
            counts = {}
            for col in self._df.columns:
                if col in counts:
                    counts[col] += 1
                    new_cols.append(f"{col}_{counts[col]}")
                else:
                    counts[col] = 0
                    new_cols.append(col)
            self._df.columns = new_cols

        # 2. Automatic identification/replacement of known junk inputs
        garbage_strings = ['?', 'n/a', 'N/A', 'NULL', 'null', ' ', '']
        self._df.replace(garbage_strings, np.nan, inplace=True)

        # 3. Logical numeric column conversion
        for col in self._df.columns:
            if self._df[col].dtype == 'object':
                try:
                    converted = pd.to_numeric(self._df[col])
                    if not converted.isna().all():
                        self._df[col] = converted
                except Exception:
                    continue

    def get_summary(self):
        """Prints high-level layout dimensions alongside a 20-row sample preview."""
        if self.df is None:
            print("Active workspace DataFrame is uninitialized."); return

        num_cols = list(self.df.select_dtypes(include=[np.number]).columns)
        cat_cols = list(self.df.select_dtypes(exclude=[np.number]).columns)

        print("="*70)
        print(f"STRUCTURAL LAYOUT SUMMARY: {self.df.shape[0]} Rows × {self.df.shape[1]} Columns")
        print(f"Numerical Feature Count ({len(num_cols)}): {num_cols}")
        print(f"Categorical Feature Count ({len(cat_cols)}): {cat_cols}")
        print("="*70)
        print("\n--- VIEWING INITIAL HEAD SAMPLES (TOP 20 INSTANCES) ---")
        display(self.df.head(20))

    def column_details(self):
        """Generates itemized type checks and missing-value diagnostics for all attributes."""
        if self.df is None: return
        details = []
        for col in self.df.columns:
            details.append({
                "Column Name": col,
                "Data Type": str(self.df[col].dtype),
                "Non-Null Count": self.df[col].notna().sum(),
                "Null Count": self.df[col].isna().sum(),
                "Unique Values": self.df[col].nunique()
            })
        print("\n--- COMPREHENSIVE ATTRIBUTE SCHEMA DETAILS ---")
        display(pd.DataFrame(details))

    def get_categorical_summary(self):
        """Extracts descriptive stats specifically for discrete categorical dimensions."""
        if self.df is None: return
        cat_df = self.df.select_dtypes(exclude=[np.number])
        if cat_df.empty:
            print("No discrete categorical configurations present in active DataFrame."); return
        print("\n--- CATEGORICAL SYSTEM PROFILE SUMMARY ---")
        display(cat_df.describe(include='all'))

    def show_missing_data(self):
        """Prints a targeted profile summarizing missing row frequencies."""
        if self.df is None: return
        missing = self.df.isna().sum()
        pct = (missing / len(self.df)) * 100
        summary_df = pd.DataFrame({"Missing Absolute": missing, "Percentage (%)": pct})
        print("\n--- MISSING ENTRY DENSITY CHECK ---")
        display(summary_df[summary_df["Missing Absolute"] > 0])

    def handle_missing_values(self, strategy: str = 'median', fill_value=None):
        """Applies configured mathematical strategies to impute missing values."""
        if self.df is None: return
        num_cols = self.df.select_dtypes(include=[np.number]).columns

        if strategy in ['mean', 'median']:
            for col in num_cols:
                val = self.df[col].mean() if strategy == 'mean' else self.df[col].median()
                self._df[col].fillna(val, inplace=True)
        elif strategy == 'mode':
            for col in self.df.columns:
                if not self.df[col].mode().empty:
                    self._df[col].fillna(self.df[col].mode()[0], inplace=True)
        elif strategy == 'constant' and fill_value is not None:
            self._df.fillna(fill_value, inplace=True)
        print(f"Missing values handled successfully using the '{strategy}' imputation strategy.")

    def remove_duplicates(self):
        """Drops duplicated row vectors from the workspace runtime environment."""
        if self.df is None: return
        init_size = len(self.df)
        self._df.drop_duplicates(inplace=True)
        print(f"Duplicate cleanup complete. Dropped {init_size - len(self.df)} identical row patterns.")

    def handle_outliers(self, columns: list = None, find_and_delete: bool = True):
        """Identifies and drops statistical outliers using IQR thresholds."""
        if self.df is None: return
        if columns is None:
            columns = list(self.df.select_dtypes(include=[np.number]).columns)

        for col in columns:
            q1 = self.df[col].quantile(0.25)
            q3 = self.df[col].quantile(0.75)
            iqr = q3 - q1
            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr

            mask = (self.df[col] < lower_bound) | (self.df[col] > upper_bound)
            outlier_count = mask.sum()
            print(f"Feature '{col}' Bounds: [{lower_bound:.2f}, {upper_bound:.2f}] -> Identified Outliers: {outlier_count}")

            if find_and_delete and outlier_count > 0:
                self._df = self.df[~mask]
                print(f"Dropped {outlier_count} outlier instances from matrix layer.")

    def delete_columns(self, input_str: str = None):
        """Drops specific columns using a manual comma-separated input string."""
        if self.df is None: return
        if not input_str:
            input_str = input("Enter column name(s) to delete (separated by commas): ")
        cols_to_drop = [c.strip() for c in input_str.split(",") if c.strip() in self.df.columns]
        self._df.drop(columns=cols_to_drop, inplace=True)
        print(f"Successfully pruned target attributes: {cols_to_drop}")

    def delete_rows(self, input_str: str = None):
        """Drops specific rows using a manual comma-separated index string."""
        if self.df is None: return
        if not input_str:
            input_str = input("Enter row index configurations to delete (separated by commas): ")
        try:
            rows_to_drop = [int(r.strip()) for r in input_str.split(",") if int(r.strip()) in self.df.index]
            self._df.drop(index=rows_to_drop, inplace=True)
            print(f"Successfully pruned target rows matching indexes: {rows_to_drop}")
        except ValueError:
            print("Aborted processing: Input contains invalid formatting.")

    def extract_normalized_numeric_data(self, method: str = 'standard') -> pd.DataFrame:
        """Transforms numeric attributes using standard, minmax, or robust scaling."""
        num_df = self.df.select_dtypes(include=[np.number]).copy()
        if num_df.empty: return num_df

        for col in num_df.columns:
            if method == 'minmax':
                min_v, max_v = num_df[col].min(), num_df[col].max()
                num_df[col] = (num_df[col] - min_v) / (max_v - min_v + 1e-9)
            elif method == 'robust':
                q1 = num_df[col].quantile(0.25)
                q3 = num_df[col].quantile(0.75)
                iqr = q3 - q1  # FIXED: Corrected reference from 'q3 - iqr' to avoid UnboundLocalError
                num_df[col] = (num_df[col] - num_df[col].median()) / (iqr + 1e-9)
            else:
                mean, std = num_df[col].mean(), num_df[col].std()
                num_df[col] = (num_df[col] - mean) / (std + 1e-9)
        return num_df

    def extract_normalized_categorical_data(self, method: str = 'onehot') -> pd.DataFrame:
        """Encodes discrete text attributes using onehot, ordinal, or uniform approaches."""
        cat_df = self.df.select_dtypes(exclude=[np.number]).copy()
        if cat_df.empty: return cat_df

        if method == 'onehot':
            return pd.get_dummies(cat_df, drop_first=True, dtype=float)
        elif method in ['ordinal', 'uniform']:
            for col in cat_df.columns:
                cat_df[col] = cat_df[col].astype('category').cat.codes.astype(float)
                if method == 'uniform':
                    min_v, max_v = cat_df[col].min(), cat_df[col].max()
                    if max_v > min_v:
                        cat_df[col] = (cat_df[col] - min_v) / (max_v - min_v)
            return cat_df
        return cat_df

    def create_normalized_data_df(self, num_method: str = 'standard', cat_method: str = 'onehot') -> pd.DataFrame:
        """Combines scaled numeric and encoded categorical features into a unified dataset."""
        num_norm = self.extract_normalized_numeric_data(method=num_method)
        cat_norm = self.extract_normalized_categorical_data(method=cat_method)
        return pd.concat([num_norm, cat_norm], axis=1)

    def plot_numerical(self, column_names: list):
        """Generates a 3-panel layout (Violin/Box, Index Scatter, Histogram) for continuous variables."""
        if self.df is None: return
        for col in column_names:
            if col not in self.df.columns: continue

            fig = make_subplots(
                rows=3, cols=1,
                subplot_titles=(f"Horizontal Violin & Box View: {col}", f"Sequence Index Scatter: {col}", f"Frequency Histogram: {col}"),
                vertical_spacing=0.12
            )
            fig.add_trace(go.Violin(x=self.df[col], box_visible=True, points='all', name="Distribution"), row=1, col=1)
            fig.add_trace(go.Scatter(x=self.df.index, y=self.df[col], mode='markers', name="Observations"), row=2, col=1)
            fig.add_trace(go.Histogram(x=self.df[col], name="Bin Counts"), row=3, col=1)

            fig.update_layout(height=800, title_text=f"Multi-Panel Continuous Evaluation: {col}", template="plotly_white", showlegend=False)
            fig.show()

    def plot_categorical(self, column_names: list):
        """Plots categorical bar charts displaying both raw counts and percentage labels."""
        if self.df is None: return
        for col in column_names:
            if col not in self.df.columns: continue
            counts = self.df[col].value_counts()
            pcts = self.df[col].value_counts(normalize=True) * 100

            fig = px.bar(
                x=counts.index, y=counts.values,
                text=[f"{v} ({p:.1f}%)" for v, p in zip(counts.values, pcts.values)],
                labels={'x': col, 'y': 'Total Absolute Frequencies'},
                title=f"Discrete Structural Interactions: {col}",
                template="plotly_white"
            )
            fig.show()

    def plot_relationship(self, x: str, y: str):
        """Automatically checks column data types to pick and render the correct visualization chart."""
        if self.df is None or x not in self.df.columns or y not in self.df.columns: return

        x_is_num = np.issubdtype(self.df[x].dtype, np.number)
        y_is_num = np.issubdtype(self.df[y].dtype, np.number)

        if x_is_num and y_is_num:
            fig = px.scatter(self.df, x=x, y=y, trendline="ols", title=f"Continuous Mapping Sequence: {x} vs {y}", template="plotly_white")
        elif not x_is_num and not y_is_num:
            df_counts = self.df.groupby([x, y]).size().reset_index(name="counts")
            fig = px.bar(df_counts, x=x, y="counts", color=y, barmode="group", title=f"Discrete Group Interactions: {x} vs {y}", template="plotly_white")
        else:
            cat_col, num_col = (x, y) if not x_is_num else (y, x)
            fig = px.box(self.df, x=cat_col, y=num_col, points="all", title=f"Mixed Distribution Profile: {num_col} by {cat_col}", template="plotly_white")
        fig.show()

    def plot_numerical_correlation(self):
        """Plots standard Pearson's r correlation coefficients across numeric features."""
        if self.df is None: return
        num_df = self.df.select_dtypes(include=[np.number])
        if num_df.empty: return

        corr_matrix = num_df.corr(method='pearson')
        fig = px.imshow(
            corr_matrix, text_auto=".2f",
            title="Pearson Correlation Matrix (Numerical vs Numerical)",
            color_continuous_scale="RdBu", color_continuous_midpoint=0, aspect="auto"
        )
        fig.show()

    def _cramers_v(self, x, y) -> float:
        """Computes Cramér's V coefficient using flat pure 1D NumPy arrays to isolate dimensions."""
        try:
            x_vec = np.asarray(self.df[x].values).flatten()
            y_vec = np.asarray(self.df[y].values).flatten()
            confusion_matrix = pd.crosstab(x_vec, y_vec)
            chi2 = chi2_contingency(confusion_matrix)[0]
            n = confusion_matrix.sum().sum()
            r, k = confusion_matrix.shape
            if n == 0 or min(r - 1, k - 1) == 0: return 0.0
            return float(np.sqrt(chi2 / (n * min(r - 1, k - 1))))
        except Exception:
            return 0.0

    def plot_categorical_correlation(self):
        """Plots a Cramér's V association matrix across categorical features."""
        if self.df is None: return
        cat_cols = self.df.select_dtypes(exclude=[np.number]).columns
        n = len(cat_cols)
        if n == 0: return

        cramers_matrix = pd.DataFrame(np.zeros((n, n)), index=cat_cols, columns=cat_cols)
        for i in range(n):
            for j in range(n):
                if i == j:
                    cramers_matrix.iloc[i, j] = 1.0
                else:
                    cramers_matrix.iloc[i, j] = self._cramers_v(cat_cols[i], cat_cols[j])

        fig = px.imshow(
            cramers_matrix, text_auto=".2f",
            title="Cramér's V Association Matrix (Categorical vs Categorical)",
            color_continuous_scale="Purples", aspect="auto"
        )
        fig.show()

    def _eta_squared(self, cat_col, num_col) -> float:
        """Calculates Eta-squared correlation via manual NumPy masking to entirely bypass scalar conversion errors."""
        if cat_col not in self.df.columns or num_col not in self.df.columns:
            return 0.0

        valid_df = self.df[[cat_col, num_col]].dropna()
        if valid_df.empty:
            return 0.0

        try:
            cat_values = np.asarray(valid_df.iloc[:, 0].values).flatten()
            num_values = np.asarray(valid_df.iloc[:, 1].values).flatten()

            unique_cats = np.unique(cat_values)
            if len(unique_cats) < 2:
                return 0.0

            groups = [num_values[cat_values == cat] for cat in unique_cats]
            groups = [g for g in groups if len(g) > 0]
            if len(groups) < 2:
                return 0.0

            f_stat, _ = f_oneway(*groups)
            f_stat = float(np.ravel(f_stat)[0])

            df_between = len(groups) - 1
            df_within = len(num_values) - len(groups)
            ss_between = f_stat * df_between

            eta = ss_between / (ss_between + df_within + 1e-9)
            return float(np.nan_to_num(eta))
        except Exception:
            return 0.0

    def plot_all_associations_heatmap(self):
        """
        Builds a comprehensive Statistical Association Heatmap matrix across all feature spaces.
        Uses Pearson's r, Cramér's V, and Eta-squared metrics based on feature type combinations.
        """
        if self.df is None: return
        cols = list(self.df.columns)
        n = len(cols)
        assoc_matrix = pd.DataFrame(np.zeros((n, n)), index=cols, columns=cols)

        for i in range(n):
            for j in range(n):
                col1, col2 = cols[i], cols[j]
                if i == j:
                    assoc_matrix.iloc[i, j] = 1.0; continue

                col1_is_num = np.issubdtype(self.df[col1].dtype, np.number)
                col2_is_num = np.issubdtype(self.df[col2].dtype, np.number)

                if col1_is_num and col2_is_num:
                    valid = self.df[[col1, col2]].dropna()
                    if len(valid) > 1:
                        v1 = np.asarray(valid.iloc[:, 0].values, dtype=float).flatten()
                        v2 = np.asarray(valid.iloc[:, 1].values, dtype=float).flatten()
                        res = pearsonr(v1, v2)
                        r_val = res[0] if isinstance(res, tuple) else getattr(res, 'statistic', 0.0)
                        assoc_matrix.iloc[i, j] = np.nan_to_num(float(np.ravel(r_val)[0]))
                    else:
                        assoc_matrix.iloc[i, j] = 0.0
                elif not col1_is_num and not col2_is_num:
                    assoc_matrix.iloc[i, j] = self._cramers_v(col1, col2)
                else:
                    c_col = col1 if not col1_is_num else col2
                    n_col = col2 if col1_is_num else col1
                    assoc_matrix.iloc[i, j] = self._eta_squared(c_col, n_col)

        fig = px.imshow(
            assoc_matrix, text_auto=".2f",
            title="Unified Association Heatmap Matrix (Pearson / Cramér's V / Eta)",
            color_continuous_scale="RdBu", color_continuous_midpoint=0, aspect="auto"
        )
        fig.show()


class PlottingMethods:
    """
    Granular chart generation helper engine[cite: 48].
    Returns HTML representations or wraps figures for flexible display formatting[cite: 49].
    """

    def get_methods_info(self) -> dict:
        """Returns metadata regarding active plotting options within this utility library[cite: 217]."""
        methods = [
            {"Method": "plot_bar_chart", "Description": "Generates grouped or stacked interactive bar charts[cite: 121]."},
            {"Method": "plot_pie_chart", "Description": "Generates variable proportion pie/donut visualizations[cite: 130]."},
            {"Method": "plot_histogram", "Description": "Plots customized distribution bin histograms[cite: 141]."},
            {"Method": "plot_heat_map", "Description": "Builds structural density heatmaps[cite: 229]."},
            {"Method": "plot_sankey_diagram", "Description": "Maps categorical workflow transitions[cite: 230]."},
            {"Method": "plot_simple_sunburst_graph", "Description": "Renders nested hierarchical tree paths[cite: 232]."}
        ]
        return {"status": "success", "response": methods}

    def display_image(self, result: dict):
        """Helper to safely extract and render HTML components inside Google Colab cells[cite: 119, 148]."""
        if result.get("status") == "success" and "html" in result:
            display(HTML(result["html"]))
        else:
            print(f"Error rendering chart asset: {result.get('message', 'Unknown Error')}")

    def plot_bar_chart(self, x: str, y: str, data: pd.DataFrame, color: str = None, barmode: str = 'group', title: str = "") -> dict:
        """Plots customizable bar visualization maps[cite: 121, 224]."""
        try:
            fig = px.bar(data, x=x, y=y, color=color, barmode=barmode, title=title, template="plotly_white")
            return {"status": "success", "html": fig.to_html(full_html=False, include_plotlyjs='cdn')}
        except Exception as e:
            return {"status": "error", "message": str(e)}

    def plot_pie_chart(self, names: str, values: str, data: pd.DataFrame, hole: float = 0.4, title: str = "") -> dict:
        """Generates variable proportion pie charts[cite: 130, 220]."""
        try:
            fig = px.pie(data, names=names, values=values, hole=hole, title=title, template="plotly_white")
            return {"status": "success", "html": fig.to_html(full_html=False, include_plotlyjs='cdn')}
        except Exception as e:
            return {"status": "error", "message": str(e)}

    def plot_histogram(self, x: str, data: pd.DataFrame, bins: list = None, title: str = "") -> dict:
        """Plots distribution intervals[cite: 141, 226]."""
        try:
            fig = px.histogram(data, x=x, title=title, template="plotly_white")
            if bins is not None:
                fig.update_traces(xbins=dict(start=bins[0], end=bins[-1], size=(bins[1]-bins[0])))
            return {"status": "success", "html": fig.to_html(full_html=False, include_plotlyjs='cdn')}
        except Exception as e:
            return {"status": "error", "message": str(e)}

    def plot_heat_map(self, values: str, index: str, columns: str, data: pd.DataFrame, title: str = "") -> dict:
        """Builds custom structural pivot heatmaps[cite: 229]."""
        try:
            pivot_df = data.pivot_table(values=values, index=index, columns=columns, aggfunc='mean')
            fig = px.imshow(pivot_df, text_auto=".2f", title=title, color_continuous_scale="YlGnBu")
            return {"status": "success", "html": fig.to_html(full_html=False, include_plotlyjs='cdn')}
        except Exception as e:
            return {"status": "error", "message": str(e)}

    def plot_sankey_diagram(self, source_column: str, target_column: str, data: pd.DataFrame, title: str = "") -> dict:
        """Generates stream flow Sankey transition mappings[cite: 230]."""
        try:
            df_counts = data.groupby([source_column, target_column]).size().reset_index(name='value')
            unique_nodes = list(set(df_counts[source_column].unique()) | set(df_counts[target_column].unique()))
            node_map = {node: i for i, node in enumerate(unique_nodes)}

            fig = go.Figure(data=[go.Sankey(
                node=dict(pad=15, thickness=20, line=dict(color="black", width=0.5), label=unique_nodes),
                link=dict(
                    source=df_counts[source_column].map(node_map),
                    target=df_counts[target_column].map(node_map),
                    value=df_counts['value']
                )
            )])
            fig.update_layout(title_text=title, font_size=12, template="plotly_white")
            return {"status": "success", "html": fig.to_html(full_html=False, include_plotlyjs='cdn')}
        except Exception as e:
            return {"status": "error", "message": str(e)}

    def plot_simple_sunburst_graph(self, path: list, values: str, data: pd.DataFrame, title: str = "") -> dict:
        """Builds nested categorical grouping wheels[cite: 232]."""
        try:
            fig = px.sunburst(data, path=path, values=values, title=title, template="plotly_white")
            return {"status": "success", "html": fig.to_html(full_html=False, include_plotlyjs='cdn')}
        except Exception as e:
            return {"status": "error", "message": str(e)}

In [ ]:
# Initialize and fetch a sample dataset (e.g., Titanic framework)
inspector = DataInspector()
inspector.df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

# Run structural checks and profile categorical attributes
inspector.get_summary()
inspector.column_details()
inspector.get_categorical_summary()
inspector.show_missing_data()

STRUCTURAL LAYOUT SUMMARY: 891 Rows × 12 Columns
Numerical Feature Count (7): ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
Categorical Feature Count (5): ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']

--- VIEWING INITIAL HEAD SAMPLES (TOP 20 INSTANCES) ---


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C



--- COMPREHENSIVE ATTRIBUTE SCHEMA DETAILS ---


,Column Name,Data Type,Non-Null Count,Null Count,Unique Values
0,PassengerId,int64,891,0,891
1,Survived,int64,891,0,2
2,Pclass,int64,891,0,3
3,Name,object,891,0,891
4,Sex,object,891,0,2
5,Age,float64,714,177,88
6,SibSp,int64,891,0,7
7,Parch,int64,891,0,7
8,Ticket,object,891,0,681
9,Fare,float64,891,0,248



--- CATEGORICAL SYSTEM PROFILE SUMMARY ---


,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Dooley, Mr. Patrick",male,347082,G6,S
freq,1,577,7,4,644



--- MISSING ENTRY DENSITY CHECK ---


,Missing Absolute,Percentage (%)
Age,177,19.865320
Cabin,687,77.104377
Embarked,2,0.224467


In [ ]:
# Clean up missing metrics, duplicates, and outliers
inspector.handle_missing_values(strategy='median')
inspector.remove_duplicates()
inspector.handle_outliers(columns=['Age', 'Fare'], find_and_delete=True)

Missing values handled successfully using the 'median' imputation strategy.
Duplicate cleanup complete. Dropped 0 identical row patterns.
Feature 'Age' Bounds: [2.50, 54.50] -> Identified Outliers: 66
Dropped 66 outlier instances from matrix layer.
Feature 'Fare' Bounds: [-25.37, 63.33] -> Identified Outliers: 107
Dropped 107 outlier instances from matrix layer.


/tmp/ipykernel_1815/1843733776.py:139: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.





In [ ]:
# Generate normalized training tables
final_ml_df = inspector.create_normalized_data_df(num_method='robust', cat_method='onehot')
print("Engineered Feature Space Matrix Shape:", final_ml_df.shape)

Engineered Feature Space Matrix Shape: (718, 1396)


In [ ]:
# Run automatic multi-variable relationship mapping charts
inspector.plot_numerical(['Age', 'Fare'])
inspector.plot_categorical(['Survived', 'Pclass'])
inspector.plot_relationship('Pclass', 'Fare')

# Run isolated and unified correlation heatmaps
inspector.plot_numerical_correlation()
inspector.plot_categorical_correlation()
inspector.plot_all_associations_heatmap()

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:573: SmallSampleWarning:

all input arrays have length 1.  f_oneway requires that at least one input has length greater than 1.

/tmp/ipykernel_1815/1843733776.py:363: SmallSampleWarning:

One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: ConstantInputWarning:

Each of the input arrays is constant; the F statistic is not defined or infinite

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: ConstantInputWarning:

Each of the input arrays is constant; the F statistic is not defined or infinite

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: ConstantInputWarning:

Each of the input arrays is constant; the F statistic is not defined or infinite

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy